In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
# 1. LOAD DATA
# ─────────────────────────────────────────
df = pd.read_csv('Gold/Elite_Potential_Hubs_Monthly_Volume_2.csv')

In [3]:
# 2. FEATURE ENGINEERING
# ─────────────────────────────────────────
df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m')
df = df.sort_values(['Outlet_ID', 'year_month']).reset_index(drop=True)

# Time features
df['month']   = df['year_month'].dt.month
df['year']    = df['year_month'].dt.year
df['quarter'] = df['year_month'].dt.quarter

# Lag features
df['lag_1'] = df.groupby('Outlet_ID')['Volume_Liters_Sum'].shift(1)
df['lag_2'] = df.groupby('Outlet_ID')['Volume_Liters_Sum'].shift(2)
df['lag_3'] = df.groupby('Outlet_ID')['Volume_Liters_Sum'].shift(3)

# Rolling features
df['rolling_mean_3'] = df.groupby('Outlet_ID')['Volume_Liters_Sum'].transform(lambda x: x.shift(1).rolling(3).mean())
df['rolling_max_3']  = df.groupby('Outlet_ID')['Volume_Liters_Sum'].transform(lambda x: x.shift(1).rolling(3).max())
df['rolling_std_3']  = df.groupby('Outlet_ID')['Volume_Liters_Sum'].transform(lambda x: x.shift(1).rolling(3).std())

# Outlet historical stats
df['outlet_max']    = df.groupby('Outlet_ID')['Volume_Liters_Sum'].transform('max')
df['outlet_mean']   = df.groupby('Outlet_ID')['Volume_Liters_Sum'].transform('mean')
df['outlet_median'] = df.groupby('Outlet_ID')['Volume_Liters_Sum'].transform('median')

# Encode categoricals
le = LabelEncoder()
df['Outlet_Size_enc']       = le.fit_transform(df['Outlet_Size'])
df['Outlet_Type_enc']       = le.fit_transform(df['Outlet_Type'])
df['Distributor_ID_enc']    = le.fit_transform(df['Distributor_ID'])
df['Seasonality_Index_enc'] = le.fit_transform(df['Seasonality_Index'])



In [4]:
# 3. TRAIN MODEL
# ─────────────────────────────────────────
features = [
    'month', 'year', 'quarter',
    'lag_1', 'lag_2', 'lag_3',
    'rolling_mean_3', 'rolling_max_3', 'rolling_std_3',
    'outlet_max', 'outlet_mean', 'outlet_median',
    'monthly_holiday_count', 'Cooler_Count',
    'Outlet_Size_enc', 'Outlet_Type_enc',
    'Distributor_ID_enc', 'Seasonality_Index_enc'
]

df_train = df.dropna(subset=features)

X = df_train[features]
y = df_train['Volume_Liters_Sum']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R2  : {r2_score(y_test, y_pred):.4f}")

MAE : 63.16
R2  : 0.9085


In [5]:
# 4. PREDICT 2026-01
# ─────────────────────────────────────────
target_month = pd.Timestamp('2026-01')
outlets = df['Outlet_ID'].unique()

# Build prediction rows for each outlet
pred_rows = []
for outlet in outlets:
    outlet_df = df[df['Outlet_ID'] == outlet].sort_values('year_month')
    last_3 = outlet_df['Volume_Liters_Sum'].values[-3:]

    row = {
        'Outlet_ID' : outlet,
        'month'     : 1,
        'year'      : 2026,
        'quarter'   : 1,
        'lag_1'     : last_3[-1] if len(last_3) >= 1 else np.nan,
        'lag_2'     : last_3[-2] if len(last_3) >= 2 else np.nan,
        'lag_3'     : last_3[-3] if len(last_3) >= 3 else np.nan,
        'rolling_mean_3' : np.mean(last_3),
        'rolling_max_3'  : np.max(last_3),
        'rolling_std_3'  : np.std(last_3),
        'outlet_max'    : outlet_df['Volume_Liters_Sum'].max(),
        'outlet_mean'   : outlet_df['Volume_Liters_Sum'].mean(),
        'outlet_median' : outlet_df['Volume_Liters_Sum'].median(),
        'monthly_holiday_count' : 3,   # Jan 2026 holiday count
        'Cooler_Count'          : outlet_df['Cooler_Count'].iloc[-1],
        'Outlet_Size_enc'       : outlet_df['Outlet_Size_enc'].iloc[-1],
        'Outlet_Type_enc'       : outlet_df['Outlet_Type_enc'].iloc[-1],
        'Distributor_ID_enc'    : outlet_df['Distributor_ID_enc'].iloc[-1],
        'Seasonality_Index_enc' : outlet_df['Seasonality_Index_enc'].iloc[-1],
    }
    pred_rows.append(row)

pred_df = pd.DataFrame(pred_rows)
pred_df['Predicted_Volume_Liters_2026_01'] = model.predict(pred_df[features])

# Save
pred_df[['Outlet_ID', 'Predicted_Volume_Liters_2026_01']].to_csv('Elite_Predictions_2026_01.csv', index=False)
print(pred_df[['Outlet_ID', 'Predicted_Volume_Liters_2026_01']].head(10))

   Outlet_ID  Predicted_Volume_Liters_2026_01
0  OUT_00010                       716.117419
1  OUT_00026                       934.669215
2  OUT_00246                       818.430891
3  OUT_00262                       791.620252
4  OUT_00303                       883.526977
5  OUT_00521                        31.448571
6  OUT_00524                        82.986062
7  OUT_00592                        53.196719
8  OUT_00841                        24.417965
9  OUT_00876                        30.087367
